# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library. All entities (record sets, fields, columns, etc.) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Instantiate Dataset object and load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We'll print a structured summary for orientation.

In [ ]:
# List record sets by their '@id' and their respective fields
print("Available Record Sets and Their Fields (all referenced by @id):\n")
record_sets = []
for rs in metadata.record_sets:
    print(f"- Record Set: {rs['@id']}, name={rs.get('name','')}, desc={rs.get('description','')}")
    record_sets.append(rs['@id'])
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    - Field: {field['@id']}, name={field.get('name','')}, dataType={field.get('dataType','')}")
    print()

if not record_sets:
    print('No record sets found! Is this an indirect schema? Attempting alternative listing...')
    # Try using files
    if hasattr(metadata, 'files'):
        for f in metadata.files:
            print(f"FileObject @id: {f['@id']}")

## 3. Data Extraction
Load data from each main record set into a DataFrame. **All references use the `@id` fields as identifiers.**

In [ ]:
# Identify main record sets and their @id
# If the previous overview didn't yield record sets, use known IDs from Croissant itself

# Example: For this dataset, let's use the main record set @id
# We'll programmatically try to select all record sets if available, else fall back to a common default
if record_sets:
    record_sets_ids = record_sets
else:
    # Use a placeholder if direct listing was not available above
    record_sets_ids = ['cr:RecordSet']

# Load one record set for demo (use the first)
selected_record_set_id = record_sets_ids[0]

dataframes = {}
for rs_id in record_sets_ids:
    print(f"Reading records from record_set @id={rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(dataframes[rs_id])} records.")
    except Exception as e:
        print(f"  Could not load {rs_id}: {e}")

# Show available columns for chosen record set
if selected_record_set_id in dataframes:
    print(f"\nColumns in {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print(f"No DataFrame loaded for {selected_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic filtering, field normalization, and grouping based on actual field `@id`s from the loaded DataFrame.

In [ ]:
# --- Example: Filtering, Normalizing, and Grouping ---

# Inspecting common mental health survey column names
from pprint import pprint
df = dataframes[selected_record_set_id]
print("Columns for analysis:")
pprint(df.columns.tolist())

# Let's pick a likely numeric field: GAD-7 score (replace with actual @id if available)
numeric_field_id = None
for col in df.columns:
    if "gad" in col.lower() or "phq" in col.lower() or "psq" in col.lower():
        numeric_field_id = col
        break
# Fallback if not found
if not numeric_field_id:
    # Use the first numeric-looking field
    for col in df.select_dtypes(include=[int, float]).columns:
        numeric_field_id = col
        break

if numeric_field_id is None:
    print('No numeric field found in the data for analysis.')
else:
    print(f"Selected numeric field for EDA (by @id/colname): {numeric_field_id}\n")
    # Filter for value > threshold
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt grouping by a categorical field: e.g., gender, education, village
    group_field = None
    for field in ["gender", "village", "education", "level_of_education", "marital_status"]:
        match = [col for col in df.columns if field in col.lower()]
        if match:
            group_field = match[0]
            break
    if group_field:
        print(f"\nGrouping results by {group_field}:")
        grouped = filtered_df.groupby(group_field)[numeric_field_id].agg(['mean','count'])
        display(grouped)
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and a boxplot by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='cornflowerblue')
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        ordered = df[group_field].value_counts().index[:6]  # up to 6 groups for readability
        sns.boxplot(x=group_field, y=numeric_field_id, data=df[df[group_field].isin(ordered)])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric field selected for visualization.')

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load AI-ready data from the FAIR² Mental Health Survey (Kilifi, Kenya) dataset, referencing all entities by their `@id`. We performed basic exploratory analysis and visualizations, paving the way for more advanced analytics and research.

_Notebook generated using schema source:_  
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`